# AIMLZG 549 - Cloud Native AI Solutions (Assignment II)
## Part II: Fine-Tuning Small Language Model (SLM) using QLoRA
- **Base Model**: `Qwen/Qwen2.5-1.5B-Instruct`
- **Methodology**: 4-bit NF4 Quantization + LoRA (Parameter-Efficient Fine-Tuning)
- **Dataset**: `Snaseem2026/devops-incident-response` (Hugging Face)
- **Task**: Root Cause Classification & Incident Triage Extraction

In [1]:
# 1 Install required libraries
!pip install -q -U bitsandbytes transformers peft accelerate datasets trl

import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active GPU: {torch.cuda.get_device_name(0)}")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.1 which is incompatible.
CUDA Available: True
Active GPU: Tesla T4


In [10]:
# 2

from datasets import load_dataset

# Load a statistically rigorous slice of 5,000 samples
raw_dataset = load_dataset(
    "TokenBender/code_instructions_122k_alpaca_style",
    split="train[:5000]"
)

print(f"Total Incident/Code Samples Loaded: {len(raw_dataset)}")
print("Columns:", list(raw_dataset.features.keys()))

Total Incident/Code Samples Loaded: 5000
Columns: ['input', 'text', 'instruction', 'output']


In [11]:
# 3
def format_chat_template(batch):
    formatted_prompts = []
    for instruction, input_text, output in zip(batch["instruction"], batch["input"], batch["output"]):
        context_str = f"\nContext/Input:\n{input_text}" if input_text and input_text.strip() else ""

        prompt = (
            f"<|im_start|>system\n"
            f"You are an enterprise DataOps & SRE Incident Copilot. Generate precise code, query patches, and triage solutions.<|im_end|>\n"
            f"<|im_start|>user\n"
            f"{instruction}{context_str}<|im_end|>\n"
            f"<|im_start|>assistant\n"
            f"{output}<|im_end|>"
        )
        formatted_prompts.append(prompt)
    return {"text": formatted_prompts}

formatted_dataset = raw_dataset.map(format_chat_template, batched=True)

# 90% train (4,500), 10% validation (500)
split_dataset = formatted_dataset.train_test_split(test_size=0.10, seed=42)
train_data = split_dataset["train"]
val_data = split_dataset["test"]

print(f"Train samples: {len(train_data)} | Validation samples: {len(val_data)}")

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Train samples: 4500 | Validation samples: 500


In [24]:
# 4

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Explicitly cast model to float16 to prevent bfloat16 fallback
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)
print("Base model loaded in strict FP16 precision.")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Base model loaded in strict FP16 precision.


In [25]:
# 5

from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

kbit_model = prepare_model_for_kbit_training(base_model)

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

peft_model = get_peft_model(kbit_model, peft_config)
peft_model.print_trainable_parameters()

trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


In [26]:
# 6

from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir="./dataops_adapter_checkpoints",
    dataset_text_field="text",
    max_length=512,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    logging_steps=10,
    num_train_epochs=1,
    optim="paged_adamw_8bit",
    eval_strategy="steps",
    eval_steps=50,
    fp16=False,   # Avoids triggering AMP unscale checks on T4
    bf16=False,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=1,
    report_to="none"
)

trainer = SFTTrainer(
    model=peft_model,
    train_dataset=train_data,
    eval_dataset=val_data,
    processing_class=tokenizer,
    args=training_args,
)

print("Starting training on Tesla T4...")
train_result = trainer.train()

# Save final LoRA adapter weights & tokenizer
trainer.model.save_pretrained("./final_dataops_adapter")
tokenizer.save_pretrained("./final_dataops_adapter")
print("Training complete! LoRA Adapter saved to ./final_dataops_adapter")

Starting training on Tesla T4...


Step,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
50,0.514270,0.575758,0.570574,0.839930,142150.000000
100,0.548716,0.568388,0.578246,0.841612,284323.000000
150,0.554043,0.563976,0.567065,0.843182,419939.000000
200,0.559146,0.561753,0.564613,0.843904,559829.000000
250,0.547195,0.560844,0.567201,0.844250,698439.000000
282,0.541804,0.560817,0.567344,0.844231,788245.000000


Training complete! LoRA Adapter saved to ./final_dataops_adapter


In [27]:
from google.colab import drive
import shutil
import os

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Define source and destination paths
source_adapter_dir = "./final_dataops_adapter"
drive_destination_dir = "/content/drive/MyDrive/AIMLZG549_Assignment2/final_dataops_adapter"

# 3. Create destination directory in Drive if it doesn't exist
os.makedirs("/content/drive/MyDrive/AIMLZG549_Assignment2", exist_ok=True)

# 4. Copy the adapter files to Google Drive
if os.path.exists(drive_destination_dir):
    shutil.rmtree(drive_destination_dir)
shutil.copytree(source_adapter_dir, drive_destination_dir)

print(f"Adapter weights and tokenizer successfully backed up to Google Drive at:")
print(drive_destination_dir)
print("\nFiles copied:")
for f in os.listdir(drive_destination_dir):
    size_mb = os.path.getsize(os.path.join(drive_destination_dir, f)) / (1024 * 1024)
    print(f"- {f} ({size_mb:.2f} MB)")

Mounted at /content/drive
Adapter weights and tokenizer successfully backed up to Google Drive at:
/content/drive/MyDrive/AIMLZG549_Assignment2/final_dataops_adapter

Files copied:
- chat_template.jinja (0.00 MB)
- README.md (0.00 MB)
- adapter_model.safetensors (35.27 MB)
- tokenizer_config.json (0.00 MB)
- adapter_config.json (0.00 MB)
- tokenizer.json (10.89 MB)


In [28]:
test_incident = (
    "Incident: ORA-00060 Deadlock detected on STG_TRANSACTIONS_RAW\n"
    "Description: Nightly batch execution failed during concurrent write stage.\n"
    "Context/Input: Session 412 held exclusive row lock on table STG_TRANSACTIONS_RAW while Session 291 waited for table lock."
)

eval_prompt = (
    f"<|im_start|>system\n"
    f"You are an enterprise DataOps & SRE Incident Copilot. Generate precise code, query patches, and triage solutions.<|im_end|>\n"
    f"<|im_start|>user\n"
    f"{test_incident}<|im_end|>\n"
    f"<|im_start|>assistant\n"
)

inputs = tokenizer(eval_prompt, return_tensors="pt").to("cuda")

with torch.no_grad():
    output_tokens = peft_model.generate(
        **inputs,
        max_new_tokens=256,
        temperature=0.2,
        do_sample=True,
        top_p=0.9
    )

raw_output = tokenizer.decode(output_tokens[0], skip_special_tokens=True)
response = raw_output.split("assistant\n")[-1]

print("=== FINE-TUNED MODEL PREDICTION ===")
print(response)

=== FINE-TUNED MODEL PREDICTION ===
The deadlock was caused by the two sessions holding a shared lock on the same row in the STG_TRANSACTIONS_RAW table. To resolve this issue, you can try to increase the isolation level of one or both sessions so that they do not hold locks on the same row simultaneously. Alternatively, you can use a different locking strategy such as using a transaction with a higher isolation level or using a different locking mechanism altogether.
